### optimised_rc

Using sample Echo State Network implementation developed by Dr. Braden Thorne to develop an optimised form of earlier ESN implementation.
- Thorne, B. (2026). method_demonstrations [Jupyter Notebook].  
--- Date:    April 2026

In [ ]:
import numpy as np

In [ ]:
#** Example code written by Dr. Braden Thorne.
#** Thorne, B. (2026). method_demonstrations [Jupyter Notebook].

def gen_reservoir_matrices(N, d, rho):
    matrices = {}
    matrices["input"] = np.random.normal(size=(N,1))
    matrices["adjacency"] = np.multiply(
        np.random.binomial(size=(N,N), p=d/N, n=1),
        np.random.normal(size=(N,N))
    )
    matrices["adjacency"] = matrices["adjacency"] * (rho/np.max(np.absolute(np.linalg.eig(matrices["adjacency"])[0])))
    matrices["bias"] = np.random.normal(size=(N,1))
    return matrices

def run_reservoir_computer(data, matrices):
    N = np.shape(matrices["adjacency"])[0]
    t = len(data)
    x = np.zeros((N,t+1))
    for i in range(t):
        x[:,i+1] = np.tanh(matrices["input"]*data[i] + np.dot(matrices["adjacency"], np.reshape(x[:,i-1], (N,1))) + matrices["bias"])[:,0]
    
    return x[:,1:]

def train_reservoir_computer(input, target, beta=0.01):
    N, t = np.shape(input)
    return np.dot(
        target, np.dot(
        np.transpose(input),
        np.linalg.pinv(np.dot(input, np.transpose(input)) - beta*np.identity(N)),
        ))

def freerun_reservoir(x0, t, matrices, readout):
    N = np.shape(matrices["adjacency"])[0]
    x = np.zeros((N,t))
    x[:,0] = x0
    for i in range(1,t):
        x[:,i] = np.tanh(matrices["input"]*np.dot(readout, x[:,i-1]) + np.dot(matrices["adjacency"], np.reshape(x[:,i-1], (N,1))) + matrices["bias"])[:,0]
    
    return np.dot(readout, x)

ESN_matrices = gen_reservoir_matrices(300, 5, 1.1)


In [ ]:
#** My implementation of the ESN using a class structure, developed upon code included above. 


class ESN:
    """
    data: data to be used
    N: number of nodes 
    d: average number of connections per node
    rho: spectral radius
    split: proportion of data to take as training  
    """
    def __init__(self, data, N, d, rho, split):
        self.data = data
        self.N = N
        self.d = d
        self.rho = rho
        self.split = split

        # random, sparse, reservoir weights:
        ## connections between nodes decided by probability of d/N
        # sampling from normal distribution provides random weight values.

        Weights = np.multiply(
            np.random.binomial(size=(self.N,self.N), p=self.d/self.N, n=1),
            np.random.normal(size=(self.N,self.N))) 

        # rescale so that rho = largest absolutel eigenvalue (maintaining echo state property)
        self.Vadj = Weights * (self.rho/np.max(np.absolute(np.linalg.eig(Weights)[0])))

        # input matrix: maps the scalar input to every node of the reservoir
        self.Vin = np.random.uniform(-1,1,size=(self.N,1))

        # bias matrix: bias added to each node's activation
        self.Vbias = np.random.uniform(-1,1,size=(self.N,))

        # assign target as one step ahead (series shifted by one)
        self.train_target = data[1:round(len(data)*split)+1]

        # portion out data for testing
        self.test_input = data[round(len(data)*split):]

    def run_reservoir(self,input_data):
        """
        drive the reservoir using the input_data and return the state at every timestep
        states = (len(input_data), N)
        """
        states = np.zeros((len(input_data), self.N))
        for i in range(1, len(input_data)):
            # feed previous state through the weights with the input and bias
            # use sigmoidal tanh to condense values
            states[i, :] = np.tanh(np.matmul(self.Vadj, states[i-1, :]) + np.matmul(self.Vin,input_data[i]) + self.Vbias)
        return states

    def train(self):
        """
        fit the output weights
        """
        # run the reservoir on the training set
        train_states = self.run_reservoir(round(len(self.data)*self.split))

        # linear readout via least squares to find Vout
        self.Vout = np.dot(
        np.linalg.pinv(train_states), self.train_target)

    #def freerun(self):

    def predict(self):
        """
        run reservoir on test data and map states to predictions
        """
        test_states = self.run_reservoir(self.test_input)
        # apply trained linear readout
        predictions = np.dot(test_states, self.Vout)

        return predictions


